# 2 · Fine-tuning de BERT

> ⚠️ **À exécuter sur Google Colab avec un GPU** : menu *Exécution → Modifier le type d'exécution → GPU T4*.
> Puis remplacer `TON_PSEUDO` dans la cellule suivante et faire *Exécution → Tout exécuter* (~10 min).

### BERT en deux mots
**BERT** (Google, 2018) est un modèle de langage qui a été **pré-entraîné** sur des milliards de mots (Wikipédia + livres) : il « comprend » déjà l'anglais et le **contexte** des mots.

Le **fine-tuning**, c'est ajouter une petite couche de sortie (86 neurones, un par tag) et **ré-entraîner l'ensemble quelques epochs** sur nos questions. BERT réutilise sa connaissance de la langue au lieu de partir de zéro.

On utilise **exactement le même découpage** train / validation / test que pour le modèle TF-IDF, pour une comparaison équitable.

In [ ]:
import os, sys
if "google.colab" in sys.modules:  # exécution sur Google Colab : on récupère le projet depuis GitHub
    REPO_URL = "https://github.com/TON_PSEUDO/stackoverflow-tag-prediction.git"  # <-- à remplacer par ton repo
    REPO_DIR = REPO_URL.rstrip("/").split("/")[-1].removesuffix(".git")
    if not os.path.exists(REPO_DIR):
        !git clone -q {REPO_URL}
    %cd {REPO_DIR}
    !pip install -q iterative-stratification
elif not os.path.exists("src"):  # exécution locale depuis le dossier notebooks/
    os.chdir("..")
print("Dossier de travail :", os.getcwd())

In [ ]:
import torch
print("GPU disponible :", torch.cuda.is_available(), "-", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "aucun (ce sera très lent !)")

In [ ]:
import json
import numpy as np
import pandas as pd
from transformers import AutoTokenizer

from src.preprocessing import prepare_dataset
from src.bert_model import BertConfig, BertTagger, train_bert
import matplotlib.pyplot as plt

BLUE, ORANGE = "#2a78d6", "#eb6834"  # couleur 1 = TF-IDF / série unique, couleur 2 = BERT
plt.rcParams.update({
    "figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False,
    "axes.edgecolor": "#b5b4ae", "axes.grid": True, "grid.color": "#e8e7e3", "grid.linewidth": 0.8,
    "axes.axisbelow": True, "axes.titleweight": "bold", "axes.titlesize": 11,
    "text.color": "#0b0b0b", "axes.labelcolor": "#52514e", "xtick.color": "#52514e", "ytick.color": "#52514e",
})

train_df, val_df, test_df, mlb = prepare_dataset()
print(f"train={len(train_df)}  val={len(val_df)}  test={len(test_df)}  tags={len(mlb.classes_)}")

## 1. Comment BERT lit un titre : la tokenisation

BERT découpe le texte en **tokens** (mots ou morceaux de mots, `##` = suite d'un mot). Chaque titre est ensuite complété ou coupé à **48 tokens** (les titres en font ~15, aucun n'est tronqué).

In [ ]:
cfg = BertConfig()
tokenizer = AutoTokenizer.from_pretrained(cfg.model_name)
print(tokenizer.tokenize("How to use C# and C++ with ASP.NET-MVC?"))
lengths = [len(tokenizer(t)["input_ids"]) for t in train_df.title]
print(f"Longueur max : {max(lengths)} tokens, moyenne : {np.mean(lengths):.1f}")

## 2. Réglages d'entraînement (choix optimisés)

| Paramètre | Valeur | Pourquoi |
|---|---|---|
| Modèle | `bert-base-uncased` | le BERT de référence (110 M paramètres) |
| Perte | `BCEWithLogits` | adaptée au multi-label : une décision oui/non par tag |
| Learning rate | 3e-5 (BERT) / 1e-3 (couche de sortie) | BERT est déjà bon : petits pas ; la couche de sortie part de zéro : grands pas |
| Warmup + décroissance linéaire | 10 % | démarrage en douceur, plus stable |
| Batch | 32 | |
| Epochs | 15 max + **early stopping** (patience 3) | on garde la meilleure epoch sur la validation, on évite le sur-apprentissage |
| Précision mixte (fp16) | activée sur GPU | ~2× plus rapide |
| Seuil de décision | optimisé sur la validation | comme pour TF-IDF |

In [ ]:
print(cfg)

## 3. Entraînement

In [ ]:
result = train_bert(train_df, val_df, test_df, mlb, cfg, save_dir="models/bert")
bert_metrics = result["metrics"]
with open("results/bert_metrics.json", "w") as f:
    json.dump(bert_metrics, f, indent=2)
bert_metrics

In [ ]:
hist = pd.DataFrame(result["history"])
fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))
axes[0].plot(hist.epoch, hist.train_loss, color=BLUE, lw=2, marker="o", ms=5)
axes[0].set(title="Perte d'entraînement", xlabel="epoch")
axes[1].plot(hist.epoch, hist.val_f1_micro, color=BLUE, lw=2, marker="o", ms=5)
axes[1].set(title="F1 micro sur la validation", xlabel="epoch")
plt.tight_layout(); plt.show()

La perte d'entraînement baisse en continu ; le F1 de validation monte puis plafonne : l'**early stopping** garde la meilleure epoch.

## 4. Comparaison TF-IDF vs BERT (sur le même jeu de test)

In [ ]:
tfidf_metrics = json.load(open("results/tfidf_metrics.json"))
keys = ["f1_micro", "f1_macro", "precision_micro", "recall_micro", "precision_at_1", "hamming_loss",
        "train_time_s", "inference_ms_per_question", "model_size_mb"]
comparison = pd.DataFrame({"TF-IDF + LogReg": [tfidf_metrics[k] for k in keys],
                           "BERT fine-tuné": [bert_metrics[k] for k in keys]}, index=keys)
comparison.to_csv("results/comparison.csv")
comparison

In [ ]:
main = ["f1_micro", "f1_macro", "precision_micro", "recall_micro", "precision_at_1"]
x = np.arange(len(main)); w = 0.38
fig, ax = plt.subplots(figsize=(10, 4))
for i, (name, color) in enumerate([("TF-IDF + LogReg", BLUE), ("BERT fine-tuné", ORANGE)]):
    vals = comparison.loc[main, name].values
    bars = ax.bar(x + (i - 0.5) * w, vals, w * 0.95, color=color, label=name)
    ax.bar_label(bars, fmt="%.2f", fontsize=8, color="#52514e", padding=2)
ax.set_xticks(x, ["F1 micro", "F1 macro", "Précision", "Rappel", "Précision@1"])
ax.set(ylim=(0, 1), title="Scores sur le jeu de test (plus haut = mieux)")
ax.grid(axis="x", visible=False); ax.legend(frameon=False, loc="upper right")
plt.tight_layout(); plt.savefig("results/comparison.png", dpi=150); plt.show()

### Interprétation
- **Qualité** : BERT comprend le contexte (synonymes, tournures) et non seulement des mots-clés ; on s'attend à ce qu'il gagne surtout en **rappel** et sur les tags génériques.
- **Coût** : BERT est ~30× plus gros, demande un GPU pour l'entraînement et est beaucoup plus lent à prédire.
- **Limite commune** : seulement ~2 900 titres très courts pour apprendre ; avec le **corps** des questions, les deux modèles progresseraient nettement.

➡️ **Du PoC à la prod** : si l'écart de qualité est faible, le TF-IDF est le choix pragmatique (léger, rapide, interprétable) ; si l'écart est significatif, BERT se justifie, éventuellement avec un modèle plus léger (DistilBERT) pour réduire la latence.

## 5. Exemples de prédictions

In [ ]:
tagger = BertTagger("models/bert")
for title in ["How to center a div with CSS?",
              "Python list comprehension with a dictionary",
              "Why is my SQL query so slow with a JOIN?",
              "NullPointerException when starting my Android activity"]:
    print(f"{title}\n   -> {tagger.predict(title)}")

## 6. Récupérer les résultats

Les fichiers à remettre dans le repo GitHub (le modèle BERT, ~440 Mo, n'est **pas** versionné) :

In [ ]:
if "google.colab" in sys.modules:
    from google.colab import files
    for f in ["results/bert_metrics.json", "results/comparison.csv", "results/comparison.png"]:
        files.download(f)